# Clublabs Intelligence - ML Model Training

This notebook demonstrates training ML models for:
1. **Escalation Prediction** - Predict if a conversation will escalate to a human agent
2. **Sentiment Classification** - Enhanced sentiment analysis beyond basic scoring
3. **Churn Risk Scoring** - Identify members at risk of cancellation

## Prerequisites
- Snowflake connection configured
- CLUBLABS_INTELLIGENCE database populated with data
- snowflake-ml-python package installed

In [ ]:
# Import required libraries
import snowflake.snowpark as snowpark
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, lit, when, avg, count, sum as sum_
from snowflake.snowpark.types import IntegerType, FloatType, StringType

from snowflake.ml.modeling.preprocessing import StandardScaler, OneHotEncoder
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.ml.modeling.metrics import accuracy_score, precision_score, recall_score, f1_score
from snowflake.ml.registry import Registry

import pandas as pd
import numpy as np

In [ ]:
# Create Snowpark session from active notebook connection
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(f"Connected to: {session.get_current_database()}.{session.get_current_schema()}")

## 1. Data Preparation

Load and prepare training data from the star schema.

In [ ]:
# Load conversation data with features for escalation prediction
conversation_features_df = session.sql("""
    SELECT 
        fc.CONVERSATION_KEY,
        fc.DURATION_SECONDS,
        fc.MESSAGE_COUNT,
        fc.BOT_MESSAGE_COUNT,
        fc.MEMBER_MESSAGE_COUNT,
        fc.SENTIMENT_SCORE,
        fc.CONFIDENCE_SCORE,
        dc.CHANNEL_NAME,
        dst.SERVICE_CATEGORY,
        dst.IS_EMERGENCY,
        dm.MEMBERSHIP_TYPE,
        dm.TENURE_MONTHS,
        dm.REGION,
        dd.IS_WEEKEND,
        HOUR(fc.START_TIMESTAMP) AS HOUR_OF_DAY,
        fc.WAS_ESCALATED AS TARGET_ESCALATED
    FROM CLUBLABS_INTELLIGENCE.ANALYTICS.FACT_CONVERSATION fc
    JOIN CLUBLABS_INTELLIGENCE.ANALYTICS.DIM_CHANNEL dc ON fc.CHANNEL_KEY = dc.CHANNEL_KEY
    JOIN CLUBLABS_INTELLIGENCE.ANALYTICS.DIM_SERVICE_TYPE dst ON fc.SERVICE_TYPE_KEY = dst.SERVICE_TYPE_KEY
    JOIN CLUBLABS_INTELLIGENCE.ANALYTICS.DIM_MEMBER dm ON fc.MEMBER_KEY = dm.MEMBER_KEY
    JOIN CLUBLABS_INTELLIGENCE.ANALYTICS.DIM_DATE dd ON fc.DATE_KEY = dd.DATE_KEY
    WHERE fc.CONVERSATION_KEY IS NOT NULL
""")

print(f"Total records: {conversation_features_df.count()}")
conversation_features_df.show(5)

In [ ]:
# Check class distribution
class_dist = conversation_features_df.group_by("TARGET_ESCALATED").count()
class_dist.show()

## 2. Feature Engineering

In [ ]:
# Define categorical and numerical columns
categorical_cols = ["CHANNEL_NAME", "SERVICE_CATEGORY", "MEMBERSHIP_TYPE", "REGION"]
numerical_cols = ["DURATION_SECONDS", "MESSAGE_COUNT", "BOT_MESSAGE_COUNT", 
                  "MEMBER_MESSAGE_COUNT", "SENTIMENT_SCORE", "CONFIDENCE_SCORE",
                  "TENURE_MONTHS", "HOUR_OF_DAY"]
boolean_cols = ["IS_EMERGENCY", "IS_WEEKEND"]

# Convert boolean columns to integers
for bool_col in boolean_cols:
    conversation_features_df = conversation_features_df.with_column(
        bool_col, 
        when(col(bool_col), lit(1)).otherwise(lit(0)).cast(IntegerType())
    )

# Target column
target_col = "TARGET_ESCALATED"

In [ ]:
# Split data into train/test sets (80/20)
train_df, test_df = conversation_features_df.random_split([0.8, 0.2], seed=42)

print(f"Training set size: {train_df.count()}")
print(f"Test set size: {test_df.count()}")

## 3. Model Training - Escalation Prediction

In [ ]:
# Create preprocessing and model pipeline
escalation_pipeline = Pipeline(
    steps=[
        ("ohe", OneHotEncoder(
            input_cols=categorical_cols,
            output_cols=[f"{c}_OHE" for c in categorical_cols],
            drop_input_cols=True
        )),
        ("scaler", StandardScaler(
            input_cols=numerical_cols + boolean_cols,
            output_cols=[f"{c}_SCALED" for c in numerical_cols + boolean_cols]
        )),
        ("classifier", XGBClassifier(
            input_cols=[f"{c}_OHE" for c in categorical_cols] + 
                       [f"{c}_SCALED" for c in numerical_cols + boolean_cols],
            label_cols=[target_col],
            output_cols=["PREDICTED_ESCALATION"],
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=42
        ))
    ]
)

print("Training escalation prediction model...")

In [ ]:
# Fit the pipeline
escalation_model = escalation_pipeline.fit(train_df)
print("Model training complete!")

In [ ]:
# Make predictions on test set
predictions_df = escalation_model.transform(test_df)

# Calculate metrics
y_true = predictions_df.select(target_col).to_pandas()[target_col]
y_pred = predictions_df.select("PREDICTED_ESCALATION").to_pandas()["PREDICTED_ESCALATION"]

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"\n=== Escalation Prediction Model Performance ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

## 4. Register Model to Snowflake ML Registry

In [ ]:
# Initialize the model registry
registry = Registry(session=session, database_name="CLUBLABS_INTELLIGENCE", schema_name="MODELS")

In [ ]:
# Log the model to the registry
model_version = registry.log_model(
    model=escalation_model,
    model_name="ESCALATION_PREDICTOR",
    version_name="v1",
    sample_input_data=train_df.limit(100),
    comment="XGBoost model to predict conversation escalation likelihood",
    metrics={
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }
)

print(f"Model registered: {model_version.model_name} version {model_version.version_name}")

## 5. Model Validation - Feature Importance

In [ ]:
# Get feature importances from the XGBoost model
try:
    xgb_model = escalation_model.stages[-1]
    feature_names = xgb_model.input_cols
    importances = xgb_model.to_sklearn().feature_importances_
    
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print("\n=== Top 10 Feature Importances ===")
    print(importance_df.head(10).to_string(index=False))
except Exception as e:
    print(f"Could not extract feature importances: {e}")

## 6. Batch Scoring Example

In [ ]:
# Score recent conversations using the registered model
batch_predictions = session.sql("""
    SELECT 
        fc.CONVERSATION_ID,
        fc.START_TIMESTAMP,
        fc.WAS_ESCALATED AS ACTUAL,
        CLUBLABS_INTELLIGENCE.MODELS.ESCALATION_PREDICTOR!PREDICT(
            fc.DURATION_SECONDS, fc.MESSAGE_COUNT, fc.BOT_MESSAGE_COUNT,
            fc.MEMBER_MESSAGE_COUNT, fc.SENTIMENT_SCORE, fc.CONFIDENCE_SCORE,
            dc.CHANNEL_NAME, dst.SERVICE_CATEGORY, dst.IS_EMERGENCY::INT,
            dm.MEMBERSHIP_TYPE, dm.TENURE_MONTHS, dm.REGION, dd.IS_WEEKEND::INT,
            HOUR(fc.START_TIMESTAMP)
        ):PREDICTED_ESCALATION AS PREDICTED
    FROM CLUBLABS_INTELLIGENCE.ANALYTICS.FACT_CONVERSATION fc
    JOIN CLUBLABS_INTELLIGENCE.ANALYTICS.DIM_CHANNEL dc ON fc.CHANNEL_KEY = dc.CHANNEL_KEY
    JOIN CLUBLABS_INTELLIGENCE.ANALYTICS.DIM_SERVICE_TYPE dst ON fc.SERVICE_TYPE_KEY = dst.SERVICE_TYPE_KEY
    JOIN CLUBLABS_INTELLIGENCE.ANALYTICS.DIM_MEMBER dm ON fc.MEMBER_KEY = dm.MEMBER_KEY
    JOIN CLUBLABS_INTELLIGENCE.ANALYTICS.DIM_DATE dd ON fc.DATE_KEY = dd.DATE_KEY
    LIMIT 10
""")
batch_predictions.show()

## Summary

This notebook demonstrated:
1. Loading training data from the Clublabs star schema
2. Feature engineering for categorical and numerical features
3. Training an XGBoost classifier for escalation prediction
4. Registering the model to Snowflake ML Registry
5. Batch scoring using the registered model

### Next Steps
- Schedule batch scoring as a Snowflake Task
- Add model monitoring for drift detection
- Train additional models (sentiment, churn risk)
- Integrate predictions into the Cortex Agent

---
## 7. Model 2: Sentiment Classification

Enhanced sentiment model with more granular categories than basic positive/neutral/negative.

In [ ]:
# Prepare data for sentiment classification (5 classes)
sentiment_df = session.sql("""
    SELECT 
        fc.CONVERSATION_KEY,
        fc.DURATION_SECONDS,
        fc.MESSAGE_COUNT,
        fc.BOT_MESSAGE_COUNT,
        fc.MEMBER_MESSAGE_COUNT,
        fc.CONFIDENCE_SCORE,
        fc.CSAT_SCORE,
        fc.WAS_ESCALATED,
        dc.CHANNEL_NAME,
        dst.SERVICE_CATEGORY,
        dst.IS_EMERGENCY,
        dm.MEMBERSHIP_TYPE,
        dm.TENURE_MONTHS,
        -- Create 5-class sentiment target
        CASE 
            WHEN fc.SENTIMENT_SCORE < -0.6 THEN 'VERY_NEGATIVE'
            WHEN fc.SENTIMENT_SCORE < -0.2 THEN 'NEGATIVE'
            WHEN fc.SENTIMENT_SCORE < 0.2 THEN 'NEUTRAL'
            WHEN fc.SENTIMENT_SCORE < 0.6 THEN 'POSITIVE'
            ELSE 'VERY_POSITIVE'
        END AS SENTIMENT_CLASS
    FROM CLUBLABS_INTELLIGENCE.ANALYTICS.FACT_CONVERSATION fc
    JOIN CLUBLABS_INTELLIGENCE.ANALYTICS.DIM_CHANNEL dc ON fc.CHANNEL_KEY = dc.CHANNEL_KEY
    JOIN CLUBLABS_INTELLIGENCE.ANALYTICS.DIM_SERVICE_TYPE dst ON fc.SERVICE_TYPE_KEY = dst.SERVICE_TYPE_KEY
    JOIN CLUBLABS_INTELLIGENCE.ANALYTICS.DIM_MEMBER dm ON fc.MEMBER_KEY = dm.MEMBER_KEY
    WHERE fc.SENTIMENT_SCORE IS NOT NULL
""")

print(f"Sentiment dataset size: {sentiment_df.count()}")
sentiment_df.group_by("SENTIMENT_CLASS").count().show()

In [ ]:
# Define features for sentiment model
sentiment_cat_cols = ["CHANNEL_NAME", "SERVICE_CATEGORY", "MEMBERSHIP_TYPE"]
sentiment_num_cols = ["DURATION_SECONDS", "MESSAGE_COUNT", "BOT_MESSAGE_COUNT", 
                     "MEMBER_MESSAGE_COUNT", "CONFIDENCE_SCORE", "CSAT_SCORE", "TENURE_MONTHS"]
sentiment_bool_cols = ["IS_EMERGENCY", "WAS_ESCALATED"]

# Convert booleans
for col_name in sentiment_bool_cols:
    sentiment_df = sentiment_df.with_column(
        col_name, when(col(col_name), lit(1)).otherwise(lit(0)).cast(IntegerType())
    )

# Split data
sentiment_train, sentiment_test = sentiment_df.random_split([0.8, 0.2], seed=42)
print(f"Sentiment train: {sentiment_train.count()}, test: {sentiment_test.count()}")

In [ ]:
# Build sentiment classification pipeline
sentiment_pipeline = Pipeline(
    steps=[
        ("ohe", OneHotEncoder(
            input_cols=sentiment_cat_cols,
            output_cols=[f"{c}_OHE" for c in sentiment_cat_cols],
            drop_input_cols=True
        )),
        ("scaler", StandardScaler(
            input_cols=sentiment_num_cols + sentiment_bool_cols,
            output_cols=[f"{c}_SCALED" for c in sentiment_num_cols + sentiment_bool_cols]
        )),
        ("classifier", XGBClassifier(
            input_cols=[f"{c}_OHE" for c in sentiment_cat_cols] + 
                       [f"{c}_SCALED" for c in sentiment_num_cols + sentiment_bool_cols],
            label_cols=["SENTIMENT_CLASS"],
            output_cols=["PREDICTED_SENTIMENT"],
            n_estimators=100,
            max_depth=5,
            learning_rate=0.1,
            random_state=42
        ))
    ]
)

print("Training sentiment classification model...")
sentiment_model = sentiment_pipeline.fit(sentiment_train)
print("Sentiment model training complete!")

In [ ]:
# Evaluate sentiment model
sentiment_preds = sentiment_model.transform(sentiment_test)
y_true_sent = sentiment_preds.select("SENTIMENT_CLASS").to_pandas()["SENTIMENT_CLASS"]
y_pred_sent = sentiment_preds.select("PREDICTED_SENTIMENT").to_pandas()["PREDICTED_SENTIMENT"]

sentiment_accuracy = accuracy_score(y_true_sent, y_pred_sent)
print(f"\n=== Sentiment Classification Model Performance ===")
print(f"Accuracy: {sentiment_accuracy:.4f}")

In [ ]:
# Register sentiment model
sentiment_version = registry.log_model(
    model=sentiment_model,
    model_name="SENTIMENT_CLASSIFIER",
    version_name="v1",
    sample_input_data=sentiment_train.limit(100),
    comment="XGBoost multi-class sentiment classifier (5 classes)",
    metrics={"accuracy": sentiment_accuracy}
)
print(f"Model registered: {sentiment_version.model_name} v{sentiment_version.version_name}")

---
## 8. Model 3: Churn Risk Scoring

Predict members at risk of cancellation based on engagement patterns.

In [ ]:
# Prepare churn risk dataset from member engagement patterns
churn_df = session.sql("""
    WITH member_stats AS (
        SELECT 
            dm.MEMBER_KEY,
            dm.MEMBERSHIP_TYPE,
            dm.REGION,
            dm.TENURE_MONTHS,
            dm.AGE_GROUP,
            dm.IS_ACTIVE,
            COUNT(fc.CONVERSATION_KEY) AS TOTAL_CONVERSATIONS,
            SUM(CASE WHEN fc.WAS_ESCALATED THEN 1 ELSE 0 END) AS ESCALATION_COUNT,
            AVG(fc.SENTIMENT_SCORE) AS AVG_SENTIMENT,
            AVG(fc.CSAT_SCORE) AS AVG_CSAT,
            MAX(dd.FULL_DATE) AS LAST_INTERACTION_DATE,
            DATEDIFF(DAY, MAX(dd.FULL_DATE), CURRENT_DATE()) AS DAYS_SINCE_LAST_INTERACTION
        FROM CLUBLABS_INTELLIGENCE.ANALYTICS.DIM_MEMBER dm
        LEFT JOIN CLUBLABS_INTELLIGENCE.ANALYTICS.FACT_CONVERSATION fc ON dm.MEMBER_KEY = fc.MEMBER_KEY
        LEFT JOIN CLUBLABS_INTELLIGENCE.ANALYTICS.DIM_DATE dd ON fc.DATE_KEY = dd.DATE_KEY
        GROUP BY dm.MEMBER_KEY, dm.MEMBERSHIP_TYPE, dm.REGION, dm.TENURE_MONTHS, dm.AGE_GROUP, dm.IS_ACTIVE
    )
    SELECT 
        MEMBER_KEY,
        MEMBERSHIP_TYPE,
        REGION,
        TENURE_MONTHS,
        AGE_GROUP,
        COALESCE(TOTAL_CONVERSATIONS, 0) AS TOTAL_CONVERSATIONS,
        COALESCE(ESCALATION_COUNT, 0) AS ESCALATION_COUNT,
        COALESCE(AVG_SENTIMENT, 0) AS AVG_SENTIMENT,
        COALESCE(AVG_CSAT, 3) AS AVG_CSAT,
        COALESCE(DAYS_SINCE_LAST_INTERACTION, 365) AS DAYS_SINCE_LAST_INTERACTION,
        -- Churn label: inactive members or simulated based on negative signals
        CASE 
            WHEN IS_ACTIVE = FALSE THEN 1
            WHEN COALESCE(AVG_SENTIMENT, 0) < -0.3 AND COALESCE(ESCALATION_COUNT, 0) >= 3 THEN 1
            WHEN COALESCE(DAYS_SINCE_LAST_INTERACTION, 365) > 180 AND COALESCE(AVG_CSAT, 3) < 3 THEN 1
            ELSE 0
        END AS CHURNED
    FROM member_stats
""")

print(f"Churn dataset size: {churn_df.count()}")
churn_df.group_by("CHURNED").count().show()

In [ ]:
# Define churn model features
churn_cat_cols = ["MEMBERSHIP_TYPE", "REGION", "AGE_GROUP"]
churn_num_cols = ["TENURE_MONTHS", "TOTAL_CONVERSATIONS", "ESCALATION_COUNT", 
                 "AVG_SENTIMENT", "AVG_CSAT", "DAYS_SINCE_LAST_INTERACTION"]

# Split data
churn_train, churn_test = churn_df.random_split([0.8, 0.2], seed=42)
print(f"Churn train: {churn_train.count()}, test: {churn_test.count()}")

In [ ]:
# Build churn prediction pipeline
churn_pipeline = Pipeline(
    steps=[
        ("ohe", OneHotEncoder(
            input_cols=churn_cat_cols,
            output_cols=[f"{c}_OHE" for c in churn_cat_cols],
            drop_input_cols=True
        )),
        ("scaler", StandardScaler(
            input_cols=churn_num_cols,
            output_cols=[f"{c}_SCALED" for c in churn_num_cols]
        )),
        ("classifier", XGBClassifier(
            input_cols=[f"{c}_OHE" for c in churn_cat_cols] + 
                       [f"{c}_SCALED" for c in churn_num_cols],
            label_cols=["CHURNED"],
            output_cols=["CHURN_PREDICTION"],
            n_estimators=100,
            max_depth=4,
            learning_rate=0.1,
            scale_pos_weight=5,  # Handle class imbalance
            random_state=42
        ))
    ]
)

print("Training churn risk model...")
churn_model = churn_pipeline.fit(churn_train)
print("Churn model training complete!")

In [ ]:
# Evaluate churn model
churn_preds = churn_model.transform(churn_test)
y_true_churn = churn_preds.select("CHURNED").to_pandas()["CHURNED"]
y_pred_churn = churn_preds.select("CHURN_PREDICTION").to_pandas()["CHURN_PREDICTION"]

churn_accuracy = accuracy_score(y_true_churn, y_pred_churn)
churn_precision = precision_score(y_true_churn, y_pred_churn)
churn_recall = recall_score(y_true_churn, y_pred_churn)
churn_f1 = f1_score(y_true_churn, y_pred_churn)

print(f"\n=== Churn Risk Model Performance ===")
print(f"Accuracy:  {churn_accuracy:.4f}")
print(f"Precision: {churn_precision:.4f}")
print(f"Recall:    {churn_recall:.4f}")
print(f"F1 Score:  {churn_f1:.4f}")

In [ ]:
# Register churn model
churn_version = registry.log_model(
    model=churn_model,
    model_name="CHURN_RISK_PREDICTOR",
    version_name="v1",
    sample_input_data=churn_train.limit(100),
    comment="XGBoost model to predict member churn risk",
    metrics={
        "accuracy": churn_accuracy,
        "precision": churn_precision,
        "recall": churn_recall,
        "f1_score": churn_f1
    }
)
print(f"Model registered: {churn_version.model_name} v{churn_version.version_name}")

---
## 9. Model Summary

| Model | Purpose | Type | Classes | Key Metric |
|-------|---------|------|---------|------------|
| ESCALATION_PREDICTOR | Predict conversation escalation | Binary | 2 | F1 Score |
| SENTIMENT_CLASSIFIER | Classify sentiment (5-level) | Multi-class | 5 | Accuracy |
| CHURN_RISK_PREDICTOR | Identify at-risk members | Binary | 2 | F1 Score |

### Registered Models in CLUBLABS_INTELLIGENCE.MODELS
- `ESCALATION_PREDICTOR!PREDICT()` - Real-time escalation scoring
- `SENTIMENT_CLASSIFIER!PREDICT()` - Enhanced sentiment analysis
- `CHURN_RISK_PREDICTOR!PREDICT()` - Member churn risk scoring

In [ ]:
# List all registered models
print("\n=== Registered Models ===")
for model in registry.get_models():
    print(f"- {model.name}")
    for version in model.versions():
        print(f"  Version: {version.version_name}, Comment: {version.comment}")